In [ ]:
import pickle

import autoroot
import matplotlib.pyplot as plt
import pandas as pd
from scipy.ndimage.filters import gaussian_filter1d

# Load the CSV file to check its content
# df = pd.read_csv("project.csv")


# Function to apply Gaussian smoothing
def smooth_gaussian(x, y):
    y_smoothed = gaussian_filter1d(y, sigma=1)
    return x, y_smoothed


def convert_tags_to_string_if_target_present(tags, target_tags):
    # Check if any of the target tags is in the tag list
    for tag in tags:
        if tag.lower() in target_tags:
            return tag.lower()


with open("project_data.pkl", "rb") as f:
    df = pickle.load(f)

In [ ]:
df = df[df["tag"].apply(lambda x: "id-1" in x)]

In [ ]:
model_list = set(["tcn", "gru", "mamba", "s4", "rnn", "lstm"])

df_original = df.copy()

rho_name = "impulse"
df["data.rho_name"] = df.config.apply(lambda x: x.get("data"))
df = df.dropna()
df["data.rho_name"] = df.config.apply(lambda x: x["data"]["rho_name"])
df["data.rho_param"] = df.config.apply(lambda x: x["data"]["rho_param"])
df["tags"] = df.config.apply(lambda x: x["run_specs"]["tags"])
df["param_total"] = df.config.apply(lambda x: x["model/params/total"])

df["val/loss"] = df["summary"].apply(lambda x: x.get("val/loss", None))
df["train/loss_epoch"] = df["summary"].apply(lambda x: x.get("train/loss_epoch", None))
df["time"] = df["summary"].apply(lambda x: x.get("_timestamp", None))

# Add a model name based on tags
df["model"] = df["tags"].apply(lambda x: convert_tags_to_string_if_target_present(x, model_list))

df["n_layers"] = df["config"].apply(lambda x: x["model"]["model"]["layers"][1]["num_layers"])
df["ckpt_path"] = df["config"].apply(lambda x: x["callbacks"]["model_checkpoint"]["dirpath"])


df = df[
    [
        "time",
        "model",
        "tags",
        "data.rho_name",
        "data.rho_param",
        "param_total",
        "train/loss_epoch",
        "val/loss",
        "n_layers",
        "ckpt_path",
        "config",
    ]
]
df.to_csv("project.csv")

# impulse

In [ ]:
df = pd.read_csv("project.csv")
df = df[df["data.rho_name"] == "impulse"]
df["data.rho_param"] = df["data.rho_param"].apply(lambda x: eval(x)["delta"])

# df = df[df["time"] > 1713619998]
# df = df[df["data.rho_param"] < 64]

grouped = (
    df.groupby(["n_layers"]).agg({"train/loss_epoch": "mean", "val/loss": "mean"}).reset_index()
)

In [ ]:
grouped

In [ ]:
plt.figure(figsize=(14, 6))


# Subplot 1: Train loss vs. Delta with log scale on y-axis and Gaussian smoothing
plt.subplot(1, 2, 1)

subset = grouped.sort_values(by="n_layers")
# x_smooth, y_smooth = subset["data.rho_param"].values, subset["train/loss_epoch"].values
x_smooth, y_smooth = smooth_gaussian(subset["n_layers"].values, subset["train/loss_epoch"].values)
plt.plot(x_smooth, y_smooth)
# plt.scatter(x_smooth, y_smooth, label=tag, s=1)

plt.xlabel("n layers")
plt.ylabel("Mean Train Loss")
plt.title("Train Loss vs. layers")
plt.yscale("log")

In [ ]:
import os
from pathlib import Path

import hydra
from omegaconf import OmegaConf

from src.data.lf.lf_datamodule import LF
from src.modules.lit_module import LitModule
from src.utils import initialize_resolvers
from src.utils.tools import get_rho_lstsq


def data_illustration(x, y):

    inputs, outputs = x, y
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))

    ax[0].plot(inputs[0, :, 0].numpy())
    ax[0].plot(outputs[0, :, 0].numpy())

    ax[1].plot(get_rho_lstsq(inputs, outputs, end_point=False)[:, 0])


initialize_resolvers()

In [ ]:
cfg = OmegaConf.load(
    "./logs/sweep/gru/impulse-multiple-layers/2024-04-25_12-12-05/.hydra/config.yaml"
)

models = []
for layer in range(1, 7):
    ckpt_path = Path(df[df["n_layers"] == layer].iloc[0]["ckpt_path"])
    ckpt_path = max((ckpt_path).glob("*.ckpt"), key=os.path.getctime)

    cfg = OmegaConf.load(
        "./logs/sweep/gru/impulse-multiple-layers/2024-04-25_12-12-05/.hydra/config.yaml"
    )
    cfg["module"]["layers"]["GRU"]["num_layers"] = layer

    model = LitModule.load_from_checkpoint(
        ckpt_path, model=hydra.utils.instantiate(cfg.module.lit_module).model
    )
    models.append(model.to("cpu"))

In [ ]:
x = LF(
    data_dir="",
    N=20,
    T=128,
    D=1,
    rho_name="impulse",
    rho_param={"delta": 64},
    input_generator="gaussian",
).dataset_construction()[:][0]

In [ ]:
yhat = models[5](x).detach()

In [ ]:
data_illustration(x, yhat)